## Шаг 2. Выбор цели, даты и группы

В рамках разведочного анализа набора данных о перелетах во время пандемии COVID-19 были выбраны следующие переменные:

*   **Целевая переменная:** `altitude_1` (высота полета в футах). 
    *Обоснование:* Это непрерывная числовая величина, которая характеризует параметры полета. Анализ её распределения позволит выявить типичные высоты, а также аномалии (например, полеты на нестандартно низких высотах во время посадки или на очень больших высотах).
*   **Столбец даты:** `day`.
    *Обоснование:* Данные охватывают период пандемии, когда авиасообщение сильно менялось. Период агрегации выбран **день**, так как это позволит детально отследить динамику изменения количества рейсов в моменты введения локдаунов и ограничений.
*   **Столбец группы:** `origin` (аэропорт вылета).
    *Обоснование:* Это категориальный признак с повторяющимися значениями (коды аэропортов IATA/ICAO). В ходе очистки данных (Лабораторная работа 1) строки с отсутствующим аэропортом вылета были удалены, поэтому в текущем наборе этот столбец не содержит пропусков. Группировка по аэропорту позволит сравнить, из каких хабов совершается больше всего рейсов и каковы их типичные параметры.

In [2]:
import pandas as pd

# Загружаем данные
df = pd.read_parquet("../data/processed/cleanfull.parquet")

# Проверяем выбранные столбцы
print("Проверка данных:")
print(f"Пропуски в altitude_1: {df['altitude_1'].isna().sum()}")
print(f"Пропуски в day: {df['day'].isna().sum()}")
print(f"Пропуски в origin: {df['origin'].isna().sum()}")  # Должно быть 0!

# Смотрим, как выглядят данные
print(df[["altitude_1", "day", "origin"]].head())

Проверка данных:
Пропуски в altitude_1: 410954
Пропуски в day: 0
Пропуски в origin: 0
   altitude_1                       day origin
0  304.799988 2019-01-01 00:00:00+00:00   YMML
1  304.799988 2019-01-01 00:00:00+00:00   YMML
2    0.000000 2019-01-01 00:00:00+00:00   YSSY
3  609.599976 2019-01-01 00:00:00+00:00   LEMD
4    0.000000 2019-01-01 00:00:00+00:00   YSSY


In [3]:
# Шаг 3. Распределение целевой переменной (altitude_1)

# Берем столбец с целевой переменной
target = df['altitude_1']

# Считаем статистику
count_total = len(target)
count_nan = target.isna().sum()
min_val = target.min()
q1 = target.quantile(0.25)
median_val = target.median()
mean_val = target.mean()
q3 = target.quantile(0.75)
max_val = target.max()

# Определяем скошенность (сравниваем среднее и медиану)
skew_text = "вправо (среднее больше медианы)" if mean_val > median_val else "влево (среднее меньше медианы)"

# Выводим результат в консоль (чтобы ты могла заполнить таблицу)
print(f"Число наблюдений: {count_total}")
print(f"Число пропусков: {count_nan}")
print(f"Минимум: {min_val:.2f}")
print(f"Первый квартиль: {q1:.2f}")
print(f"Медиана: {median_val:.2f}")
print(f"Среднее: {mean_val:.2f}")
print(f"Третий квартиль: {q3:.2f}")
print(f"Максимум: {max_val:.2f}")
print(f"Скошенность: {skew_text}")

Число наблюдений: 7248316
Число пропусков: 410954
Минимум: 0.00
Первый квартиль: 0.00
Медиана: 304.80
Среднее: 424.27
Третий квартиль: 609.60
Максимум: 5791.20
Скошенность: вправо (среднее больше медианы)


### Таблица 1 - Сводка целевой переменной

| Показатель | Значение |
| :--- | :--- |
| Число наблюдений | 7248316 |
| Число пропусков | 410 954 |
| Минимум | 0 |
| Первый квартиль | 0 |
| Медиана | 304.80 |
| Среднее | 424.27 |
| Третий квартиль | 609.60 |
| Максимум | 5791.20 |
| Скошенность | вправо (среднее больше медианы) |

Пояснение: «Распределение высоты полета скошено вправо. Это означает, что большинство рейсов летает на средних высотах, но есть небольшая часть полетов на очень больших высотах, которые тянут среднее значение вверх, делая его больше медианы».

In [5]:
# Шаг 4. Динамика по датам

import pandas as pd

# 1. Оставляем только строки, где есть дата, и группируем по дням
df_daily = df.dropna(subset=['day']).groupby('day').agg(
    count=('altitude_1', 'size'),          # Число записей
    median_target=('altitude_1', 'median'), # Медиана цели
    mean_target=('altitude_1', 'mean')      # Среднее цели
).reset_index()

# 2. Создаем полный календарь (от мин до макс даты), чтобы вставить пустые дни
min_date = df_daily['day'].min()
max_date = df_daily['day'].max()
full_dates = pd.date_range(start=min_date, end=max_date, freq='D')

# Переиндексируем по полному календарю (пустые дни станут NaN)
df_daily = df_daily.set_index('day').reindex(full_dates).reset_index()
df_daily.rename(columns={'index': 'day'}, inplace=True)

# 3. Заполняем пустые периоды нулями для числа записей
df_daily['count'] = df_daily['count'].fillna(0).astype(int)

# 4. Выводим данные: Начало, Середина, Конец
print("--- НАЧАЛО РЯДА ---")
print(df_daily.head(5).to_string(index=False))

print("\n--- СЕРЕДИНА РЯДА ---")
middle_idx = len(df_daily) // 2
print(df_daily.iloc[middle_idx-2:middle_idx+3].to_string(index=False))

print("\n--- КОНЕЦ РЯДА ---")
print(df_daily.tail(5).to_string(index=False))

--- НАЧАЛО РЯДА ---
                      day  count  median_target  mean_target
2019-01-01 00:00:00+00:00  28555     304.799988   406.955780
2019-01-02 00:00:00+00:00  36210     304.799988   426.363037
2019-01-03 00:00:00+00:00  38172     304.799988   442.004639
2019-01-04 00:00:00+00:00  40450     304.799988   457.143677
2019-01-05 00:00:00+00:00  35983     304.799988   468.234650

--- СЕРЕДИНА РЯДА ---
                      day  count  median_target  mean_target
2019-03-30 00:00:00+00:00  33411     304.799988   412.229340
2019-03-31 00:00:00+00:00  35064     304.799988   412.777191
2019-04-01 00:00:00+00:00  38950     304.799988   413.472778
2019-04-02 00:00:00+00:00  38280     304.799988   426.315918
2019-04-03 00:00:00+00:00  44142     304.799988   423.220886

--- КОНЕЦ РЯДА ---
                      day  count  median_target  mean_target
2019-06-26 00:00:00+00:00  50862     304.799988   432.767639
2019-06-27 00:00:00+00:00  47633     304.799988   438.504364
2019-06-28 00:00:00+00

C:\Users\Jesayz\AppData\Local\Temp\ipykernel_29972\3727342268.py:19: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_daily.rename(columns={'index': 'day'}, inplace=True)


### Таблица 2 - Динамика по периодам

| Период (Дата) | Число записей | Медиана цели | Среднее цели |
| :--- | :--- | :--- | :--- |
| 2019-01-01 | 28555 | 304.799988 | 406.955780 |
| 2019-01-05 | 35983 | 304.799988 | 468.234650 |
| 2019-04-01 | 38950 | 304.799988 | 413.472778 |
| 2019-06-26 | 50862 | 304.799988 | 432.767639 |
| 2019-06-30 | 44507 | 304.799988 | 412.634796 |